# Part C Schritt fuer Schritt

Dieses Notebook zeigt die Road-Segmentation aus `aibip.part_c` mit sichtbaren Zwischenergebnissen.

Part C macht aus einem Rohbild eine Strassenmaske:
- Part-B-Preprocessing anwenden
- feste MobileSAM-Prompts auf das vorbereitete Bild setzen
- drei SAM-Kandidaten erzeugen
- den besten Kandidaten mit der Anker-/Near-Field-Heuristik auswaehlen
- Maske bereinigen
- Allowed Region fuer Part D erzeugen

Wichtig: Fuer den SAM-Schritt muss `models/mobile_sam_vit_t.pt` vorhanden sein. Falls nicht, fuehre im Terminal aus: `.\.venv\Scripts\python.exe -m aibip.fetch_models`.

In [ ]:
from pathlib import Path
import sys
import time

# Notebook kann aus notebooks/ gestartet werden; Repo-Root in sys.path aufnehmen.
ROOT = Path.cwd()
if not (ROOT / "aibip").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import cv2
import numpy as np

from aibip import config, imgio
from aibip.part_b import preprocess
from aibip.part_c import (
    _prompts,
    _get_predictor,
    _torch_device,
    select_candidate,
    clean_mask,
    allowed_region,
    mask_stats,
    flag_stats_outliers,
    _overlay,
)

# aibip.part_b nutzt fuer CLI-Figuren kurz Agg; im Notebook danach inline.
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass

import matplotlib.pyplot as plt
from IPython.display import display

plt.rcParams["figure.figsize"] = (14, 7)
plt.rcParams["axes.titlesize"] = 10

print("Repo:", ROOT)
print("Device fuer MobileSAM:", _torch_device())
print("Arbeitsbreite aus Part B:", config.WORK_WIDTH)

## 1. Beispielbilder laden

Die Beispielbilder werden aus `data/mini` geladen.

In [ ]:
EXAMPLE_PATHS = [
    Path("data/mini/WE1MY5SK.jpg"),
    Path("data/mini/WE1MY5SL.jpg"),
    Path("data/mini/WE1MY5SP.jpg"),
    Path("data/mini/WE1MY5SN.jpg"),
    Path("data/mini/WE1MY5SM.jpg"),
    Path("data/mini/WE1MY5SR.jpg"),
    Path("data/mini/WE1MY5SQ.jpg"),
    Path("data/mini/WE1MY5SO.jpg"),
    Path("data/mini/WE1MY5SS.jpg"),
    Path("data/mini/WE1MY5ST.jpg"),
]

paths = [p for p in EXAMPLE_PATHS if p.exists()]
if not paths:
    fallback = ROOT / "data" / "mini"
    paths = imgio.list_images(fallback)[:10] if fallback.exists() else []
if not paths:
    raise FileNotFoundError("Keine Beispielbilder gefunden. Lege Bilder in data/mini oder passe EXAMPLE_PATHS an.")

raw = {p.stem: imgio.load_image(p) for p in paths}
print(f"Gefundene Bilder: {len(raw)}")
for p in paths:
    print("-", p)

In [ ]:
def bgr_to_rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def show_images(items, cols=3, title=None):
    rows = int(np.ceil(len(items) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 3.4 * rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (name, img) in zip(axes.ravel(), items):
        ax.imshow(bgr_to_rgb(img))
        ax.set_title(name)
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    display(fig)
    plt.close(fig)

show_images([(name, imgio.resize(img, (384, 512))) for name, img in list(raw.items())[:6]], cols=3, title="Rohbilder")

## 2. Part-B-Preprocessing als Eingang fuer Part C

MobileSAM sieht nicht das volle 8192x6144-Bild, sondern den vorbereiteten Arbeitsbereich aus Part B. Dadurch sind Himmel und Haube groesstenteils raus und die Prompts liegen stabiler.

In [ ]:
processed = {name: preprocess(img) for name, img in raw.items()}
for name, img in list(processed.items())[:3]:
    print(name, img.shape)

show_images([(name, imgio.resize(img, (307, 1024))) for name, img in list(processed.items())[:6]], cols=2, title="Input fuer Part C: preprocess(raw)")

## 3. Prompt-Punkte anzeigen

Gruene Punkte sind positive Prompts: Dort soll Strasse sein. Rote Punkte sind negative Prompts: Dort soll eher keine Strasse sein, z.B. Himmel, Randbereiche oder Haubenrest.

In [ ]:
def draw_prompts(proc):
    out = proc.copy()
    h, w = out.shape[:2]
    points, labels = _prompts(h, w)
    for (x, y), label in zip(points.astype(int), labels):
        color = (0, 255, 0) if label == 1 else (0, 0, 255)
        cv2.circle(out, (int(x), int(y)), 13, color, -1)
        cv2.circle(out, (int(x), int(y)), 16, (255, 255, 255), 2)
    return out

prompt_views = [(name, imgio.resize(draw_prompts(proc), (307, 1024))) for name, proc in list(processed.items())[:6]]
show_images(prompt_views, cols=2, title="MobileSAM-Prompts: gruen=Strasse, rot=nicht Strasse")

## 4. Modell-Datei pruefen

Diese Zelle laedt noch kein Modell. Sie prueft nur, ob die Gewichtedatei existiert. Wenn nicht, bitte erst im Terminal laden.

In [ ]:
model_path = ROOT / config.MODELS_DIR / "mobile_sam_vit_t.pt"
print("Model path:", model_path)
if not model_path.exists():
    raise FileNotFoundError(
        "MobileSAM-Gewicht fehlt. Fuehre im Terminal aus:\n"
        ".\\.venv\\Scripts\\python.exe -m aibip.fetch_models"
    )
print("OK: Modell-Datei vorhanden", f"({model_path.stat().st_size / 1e6:.1f} MB)")

## 5. Ein Bild durch MobileSAM schicken

Beim ersten Ausfuehren kann diese Zelle etwas dauern, weil das Modell geladen wird. Auf CPU ist es deutlich langsamer als auf MPS/CUDA.

In [ ]:
def sam_candidates_for_image(raw_img):
    proc = preprocess(raw_img)
    h, w = proc.shape[:2]
    points, labels = _prompts(h, w)
    predictor = _get_predictor()

    t0 = time.perf_counter()
    predictor.set_image(proc[:, :, ::-1])
    t_set = time.perf_counter() - t0

    t0 = time.perf_counter()
    masks, scores, logits = predictor.predict(
        point_coords=points,
        point_labels=labels,
        multimask_output=True,
    )
    t_predict = time.perf_counter() - t0

    selected_candidate, selected_idx = select_candidate(masks, scores, h, w)
    cleaned = clean_mask(selected_candidate)
    allowed = allowed_region(cleaned)
    stats = mask_stats(selected_candidate, scores[selected_idx])
    return {
        "proc": proc,
        "points": points,
        "labels": labels,
        "masks": masks.astype(bool),
        "scores": scores,
        "selected_idx": selected_idx,
        "selected": selected_candidate.astype(bool),
        "cleaned": cleaned,
        "allowed": allowed,
        "stats": stats,
        "t_set": t_set,
        "t_predict": t_predict,
    }

first_name = list(raw.keys())[0]
result = sam_candidates_for_image(raw[first_name])
print("Bild:", first_name)
print(f"set_image: {result['t_set']:.2f}s, predict: {result['t_predict']:.2f}s")
print("SAM scores:", [round(float(s), 4) for s in result["scores"]])
print("selected_idx:", result["selected_idx"])
print("mask_stats:", {k: round(v, 4) if isinstance(v, float) else v for k, v in result["stats"].items()})

## 6. Die drei SAM-Kandidaten ansehen

SAM gibt bei `multimask_output=True` mehrere plausible Masken zurueck. Der Code nimmt nicht blind den hoechsten SAM-Score, sondern bevorzugt die Maske, die den unteren Strassenanker trifft und den unteren Strassenbereich gut abdeckt.

In [ ]:
def mask_overlay(proc, mask, outside_alpha=0.35):
    out = proc.copy()
    outside = ~mask
    red = np.array([0, 0, 255], dtype=np.float32)
    out[outside] = (out[outside] * (1 - outside_alpha) + red * outside_alpha).astype(np.uint8)
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(out, contours, -1, (0, 0, 255), 2)
    return out

candidate_items = []
for i, (mask, score) in enumerate(zip(result["masks"], result["scores"])):
    label = f"candidate {i} score={float(score):.3f}"
    if i == result["selected_idx"]:
        label += " SELECTED"
    candidate_items.append((label, imgio.resize(mask_overlay(result["proc"], mask), (307, 1024))))

show_images(candidate_items, cols=1, title="SAM-Kandidaten: rot markiert ist ausserhalb der Maske")

## 7. Auswahlheuristik sichtbar machen

Die Auswahl passiert mit drei Kriterien:

1. Trifft die Maske den Bottom-Center-Anker bei 80 Prozent der Hoehe?
2. Wie viel vom unteren Streifen 70 bis 85 Prozent ist abgedeckt?
3. SAM-Score als Tie-Breaker.

In [ ]:
h, w = result["proc"].shape[:2]
anchor_rc = (int(0.80 * h), int(0.50 * w))
rows = []
for i, (mask, score) in enumerate(zip(result["masks"], result["scores"])):
    strip_cov = float(mask[int(0.70 * h):int(0.85 * h)].mean())
    rows.append({
        "candidate": i,
        "anchor_ok": bool(mask[anchor_rc]),
        "strip_cov": strip_cov,
        "sam_score": float(score),
        "selected": i == result["selected_idx"],
    })

for row in rows:
    print(row)

view = result["proc"].copy()
cv2.circle(view, (anchor_rc[1], anchor_rc[0]), 16, (0, 255, 255), -1)
cv2.rectangle(view, (0, int(0.70 * h)), (w - 1, int(0.85 * h)), (255, 0, 0), 3)
show_images([("gelb=Anker, blau=Near-Field-Streifen", imgio.resize(view, (307, 1024)))], cols=1)

## 8. Maske bereinigen

`clean_mask()` nimmt die groesste zusammenhaengende Komponente und entfernt kleine Stoerungen mit Morphological Opening. Das macht aus dem SAM-Kandidaten eine stabilere Strassenmaske.

In [ ]:
selected_view = mask_overlay(result["proc"], result["selected"])
cleaned_view = mask_overlay(result["proc"], result["cleaned"])
show_images([
    ("ausgewaehlter SAM-Kandidat", imgio.resize(selected_view, (307, 1024))),
    ("nach clean_mask", imgio.resize(cleaned_view, (307, 1024))),
], cols=1, title="Maskenbereinigung")

## 9. Allowed Region fuer Part D

`allowed_region()` erodiert die Strassenmaske um `PATCH//2`. Dadurch darf ein 128x128-Patch spaeter nur dort zentriert werden, wo er komplett innerhalb der Strasse liegt. Das verhindert, dass Bordstein, Vegetation oder Maskenraender in die Trainingspatches geraten.

In [ ]:
def allowed_overlay(proc, mask, allowed):
    out = mask_overlay(proc, mask, outside_alpha=0.25)
    green = np.array([0, 255, 0], dtype=np.float32)
    out[allowed] = (out[allowed] * 0.45 + green * 0.55).astype(np.uint8)
    return out

allow_view = allowed_overlay(result["proc"], result["cleaned"], result["allowed"])
print("Maskenflaeche:", round(float(result["cleaned"].mean()), 4))
print("Allowed-Region-Flaeche:", round(float(result["allowed"].mean()), 4))
show_images([("rot=ausserhalb Maske, gruen=erlaubte Patch-Zentren", imgio.resize(allow_view, (307, 1024)))], cols=1)

## 10. Mehrere Beispielbilder segmentieren

Diese Zelle laeuft ueber mehrere Bilder. Auf CPU kann das etwas dauern. Fuer einen schnellen Test kannst du `N = 3` lassen.

In [ ]:
N = min(3, len(raw))  # auf CPU erstmal klein halten; bei Bedarf erhoehen
batch_results = {}
records = []

for name, img in list(raw.items())[:N]:
    t0 = time.perf_counter()
    res = sam_candidates_for_image(img)
    batch_results[name] = res
    records.append({"image_id": name, **res["stats"]})
    print(f"{name}: selected={res['selected_idx']} area={res['stats']['area_frac']:.3f} strip={res['stats']['strip_cov']:.3f} time={time.perf_counter() - t0:.1f}s")

flags = flag_stats_outliers(records)
print("\nFlags in dieser kleinen Stichprobe:")
if flags:
    for flag in flags:
        print(flag)
else:
    print("Keine robusten Masken-Ausreisser.")

In [ ]:
overlay_items = []
for name, res in batch_results.items():
    overlay_items.append((name, imgio.resize(_overlay(res["proc"], res["cleaned"]), (307, 1024))))

show_images(overlay_items, cols=1, title="Finale Part-C-Masken: rot ist ausserhalb der Strassenmaske")

## 11. Ergebnisse speichern

Diese Zelle speichert Masken und Overlays fuer die im Notebook gerechneten Bilder unter `outputs/notebook_part_c/`.

In [ ]:
out_dir = ROOT / "outputs" / "notebook_part_c"
mask_dir = out_dir / "masks"
overlay_dir = out_dir / "overlays"
mask_dir.mkdir(parents=True, exist_ok=True)
overlay_dir.mkdir(parents=True, exist_ok=True)

for name, res in batch_results.items():
    imgio.save_image(mask_dir / f"{name}.png", res["cleaned"].astype(np.uint8) * 255)
    imgio.save_image(overlay_dir / f"{name}.jpg", _overlay(res["proc"], res["cleaned"]))

print("Gespeichert unter:", out_dir)

## 12. Offizielle Part-C-Kommandos

Wenn du spaeter nicht im Notebook, sondern im Projektmodus arbeiten willst:

```powershell
.\.venv\Scripts\python.exe -m aibip.fetch_models
.\.venv\Scripts\python.exe -m aibip.part_c benchmark data\mini --n 4
.\.venv\Scripts\python.exe -m aibip.part_c run data\mini
.\.venv\Scripts\python.exe -m aibip.part_c qc data\mini --n 12
```

Die wichtigsten Outputs sind:
- `outputs/masks/*.png`
- `outputs/mask_stats.csv`
- `outputs/flags.csv`
- `outputs/qc/overlays.jpg`